In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

# Add parent directory to path to import 'deep_ecg' modules
sys.path.append(os.path.abspath(os.path.join('..')))

from load_dataset import load_heartprint_dataset
from run import (
    run_closed_set_identification, 
    run_subject_disjoint_identification,
    run_verification,
    run_subject_disjoint_verification,
    run_cross_session_identification,
    run_cross_session_verification
)
from models import DeepECG, ResNet1D
from visualizations import Visualizer

import warnings
warnings.filterwarnings("ignore")

print("Framework loaded.")
# print(f"Available HeartPrint Sessions: {SESSIONS}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

Framework loaded.
Running on: cpu


In [2]:
# --- TASKS 1-3: Random Split (Base) ---
print("Loading HeartPrint (All Data) for Random Split...")

# Mode doesn't strictly matter for load_all_sessions(), but we stick to standard
loader = load_heartprint_dataset(num_beats=3, enrollment_mode='standard', cleanup_zip=False)
x_all, y_all = loader.load_all_sessions()

print(f"Data Loaded: {x_all.shape} samples, {len(np.unique(y_all))} subjects")

# Run Standard Benchmarks
print("\n=== TASK 1: Closed-Set Identification ===")
run_closed_set_identification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n=== TASK 2: Verification ===")
run_verification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n=== TASK 3: Subject-Disjoint Verification ===")
run_subject_disjoint_verification(x_all, y_all, DeepECG, epochs=2, device=device)

Loading HeartPrint (All Data) for Random Split...


Processing all sessions: 100%|██████████| 199/199 [01:41<00:00,  1.95it/s]


Data Loaded: (16369, 150) samples, 199 subjects

=== TASK 1: Closed-Set Identification ===

[TASK] Closed-Set Identification on cpu
    Epoch 001 | Loss: 4.6128
    Epoch 002 | Loss: 3.5568
[RESULT] Closed-Set Accuracy: 0.1863

=== TASK 2: Verification ===

[TASK] Verification (Random Split / Closed-Set) on cpu
[INFO] Phase 1: Training feature extractor (Learning Identity)...
    Epoch 001 | Loss: 4.7265
    Epoch 002 | Loss: 3.5855
[INFO] Phase 2: Computing EER using 'balanced' sampling...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BALANCED | EER: 0.1834 | AUC: 0.9012

=== TASK 3: Subject-Disjoint Verification ===

[TASK] Subject-Disjoint Verification (Open-Set) on cpu
[INFO] Splitting: 139 Training Subjects vs 60 Test Subjects
[INFO] Phase 1: Training on Known Subjects...
    Epoch 001 | Loss: 4.2843
    Epoch 002 | Loss: 3.2501
[INFO] Phase 2: Computing EER on 60 Unseen Subjects...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BAL

{'eer': 0.17480000000059828, 'auc': 0.9048227799999999}

In [4]:
# --- TASK 4: Biometric Regimes Evaluation ---
print("\n=== TASK 4: Biometric Regimes ===")

# A. STANDARD (Long-Term)
# S1 -> S2
print("\n[A] Loading Standard Regime (S1 -> S2)...")
loader_std = load_heartprint_dataset(num_beats=3, enrollment_mode='standard', cleanup_zip=False)
x_std_enr, y_std_enr = loader_std.load_session("Session_1")
x_std_prb, y_std_prb = loader_std.load_session("Session_2")
print(f"    Standard Shapes: Enroll {x_std_enr.shape}, Probe {x_std_prb.shape}")

print("    Running Verification...")
# run_cross_session_verification(x_std_enr, y_std_enr, x_std_prb, y_std_prb, DeepECG, epochs=15, device=device, visualize=False)


# B. REVERSE (Long-Term)
# S2 -> S1
print("\n[B] Loading Reverse Regime (S2 -> S1)...")
loader_rev = load_heartprint_dataset(num_beats=3, enrollment_mode='reverse', cleanup_zip=False)
x_rev_enr, y_rev_enr = loader_rev.load_session("Session_1")
x_rev_prb, y_rev_prb = loader_rev.load_session("Session_2")
print(f"    Reverse Shapes: Enroll {x_rev_enr.shape}, Probe {x_rev_prb.shape}")

print("    Running Verification...")
# run_cross_session_verification(x_rev_enr, y_rev_enr, x_rev_prb, y_rev_prb, DeepECG, epochs=15, device=device, visualize=False)


# C. STATE ROBUSTNESS (Reading Task)
# S1 -> S3R
print("\n[C] Loading State Robustness (S1 -> S3R Reading)...")
loader_state = load_heartprint_dataset(num_beats=3, enrollment_mode='state_robustness', cleanup_zip=False)
x_st_enr, y_st_enr = loader_state.load_session("Session_1")
x_st_prb, y_st_prb = loader_state.load_session("Session_2")
print(f"    State Rob Shapes: Enroll {x_st_enr.shape}, Probe {x_st_prb.shape}")

print("    Running Verification...")
# run_cross_session_verification(x_st_enr, y_st_enr, x_st_prb, y_st_prb, DeepECG, epochs=15, device=device, visualize=True)


# D. VERY LONG TERM (Maximal Interval)
# S1 -> S3L
print("\n[D] Loading VLT Regime (S1 -> S3L Long Interval)...")
loader_vlt = load_heartprint_dataset(num_beats=3, enrollment_mode='vlt', cleanup_zip=False)
x_vlt_enr, y_vlt_enr = loader_vlt.load_session("Session_1")
x_vlt_prb, y_vlt_prb = loader_vlt.load_session("Session_2")
print(f"    VLT Shapes: Enroll {x_vlt_enr.shape}, Probe {x_vlt_prb.shape}")

print("    Running Verification...")
# run_cross_session_verification(x_vlt_enr, y_vlt_enr, x_vlt_prb, y_vlt_prb, DeepECG, epochs=15, device=device, visualize=True)


=== TASK 4: Biometric Regimes ===

[A] Loading Standard Regime (S1 -> S2)...


Processing session2 signals: 100%|██████████| 199/199 [00:44<00:00,  4.42it/s]


    Standard Shapes: Enroll (8281, 150), Probe (8088, 150)
    Running Verification...

[B] Loading Reverse Regime (S2 -> S1)...


Processing session1 signals: 100%|██████████| 199/199 [00:47<00:00,  4.19it/s]


    Reverse Shapes: Enroll (8088, 150), Probe (8281, 150)
    Running Verification...

[C] Loading State Robustness (S1 -> S3R Reading)...


Processing session3r signals: 100%|██████████| 109/109 [00:38<00:00,  2.83it/s]


    State Rob Shapes: Enroll (8281, 150), Probe (6575, 150)
    Running Verification...

[D] Loading VLT Regime (S1 -> S3L Long Interval)...


Processing session3l signals: 100%|██████████| 78/78 [00:23<00:00,  3.33it/s]

    VLT Shapes: Enroll (8281, 150), Probe (4075, 150)
    Running Verification...


In [ ]:
# --- TASK 5: Blind Segmentation (Standard Regime) ---
print("\n=== TASK 5: Blind Segmentation (Standard Regime) ===")

blind_params = {
    'mode': 'blind',
    'window_len': 5.0,  
    'stride': 2.0,      
    'bandpass': True,
    'normalize': 'zscore'
}

# Use 'standard' mode (S1 -> S2)
loader_blind = load_heartprint_dataset(
    num_beats=1, 
    enrollment_mode='standard', 
    preprocessing_params=blind_params, 
    cleanup_zip=False
)

x_blind_enr, y_blind_enr = loader_blind.load_session("Session_1")
x_blind_prb, y_blind_prb = loader_blind.load_session("Session_2")

print(f"Blind Data Shapes: Enroll {x_blind_enr.shape}, Probe {x_blind_prb.shape}")

print("\n[Blind] Identification")
run_cross_session_identification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=10, device=device)

print("\n[Blind] Verification")
run_cross_session_verification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=10, device=device, visualize=True)